# Making Agents Reliable

Add requirements validation and Guardian safety checks to a ReACT tool-using agent.

This tutorial shows how to build a tool-using agent with Mellea and progressively add reliability layers: output requirements, retry budgets, and Guardian safety checks that detect harmful or off-topic responses before they reach your users.

By the end you will have covered:

* Building a simple tool-using agent with `@tool` decorated functions
* Adding output requirements with `req()` and validation functions
* Inspecting failures and handling retry budgets with `RejectionSamplingStrategy`
* Adding Guardian harm detection with `GuardianCheck` and `GuardianRisk` categories
* Optimizing Guardian performance with shared backends
* Implementing groundedness checks with retrieved context
* Building a complete ReACT agent with Guardian validation

Prerequisites: Tutorial 01 complete, Mellea installed (`uv add mellea`), Ollama running locally with `granite4:micro` downloaded.

---

## Step 1: A simple tool-using agent

Start with two tools — a search stub and a calculator — and wire them into an `instruct()` call.

**Key Concepts:**

- `@tool` decorator converts Python functions into LLM-callable tools
- Tools are defined with docstrings that describe their purpose to the LLM
- The `tool` parameter in `instruct()` accepts a list of tool functions
- `ModelOption.TOOLS` enables tool-calling mode in the backend
- The LLM can invoke tools and use their results to answer questions
- Each tool function should have clear type hints and a descriptive docstring

In [ ]:
import mellea
from mellea.backends import ModelOption, tool


# Define a search tool using the @tool decorator
# The docstring tells the LLM what this tool does
@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic.
    
    Args:
        query: The search query.
    """
    # In production, this would call a real search API
    # For now, return a mock result
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


# Define a calculator tool with input validation
@tool
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression and return the result.
    
    Args:
        expression: An arithmetic expression, e.g. '12 * 7 + 3'.
    """
    # Define allowed characters for safe evaluation
    allowed = set("0123456789 +-*/(). ")
    
    # Validate input to prevent code injection
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    
    # Safely evaluate the expression
    return str(eval(expression))  # noqa: S307 - controlled eval with validation


# Initialize Mellea session
m: mellea.MelleaSession = mellea.start_session()

# Call instruct with tools enabled
# The LLM can now invoke web_search and calculate as needed
response = m.instruct(
    "What is Mellea, and how many characters are in the word 'Mellea'?",
    model_options={ModelOption.TOOLS: [web_search, calculate]},
)

print(str(response))

# Output will vary — LLM responses depend on model and temperature.

## Step 2: Adding output requirements

Require the agent to format its answer as a short structured response.

**Key Concepts:**

- Requirements enforce output constraints using the Instruct-Validate-Repair (IVR) pattern
- `req()` creates a Requirement object with optional custom validation
- Plain-English requirements are validated using LLM-as-a-judge by default
- `simple_validate()` wraps Python functions for deterministic validation
- Deterministic checks (like word count) are much faster than LLM-based validation
- Mix both approaches: use deterministic validation where possible, LLM-as-a-judge for subjective checks
- Failed requirements trigger automatic retry with the failure reason embedded in the repair request

The word-count requirement runs deterministically. The "answer both questions" requirement falls back to LLM-as-a-judge. If either fails, Mellea retries with the failure reason embedded in the repair request.

In [ ]:
import mellea
from mellea.backends import ModelOption, tool
from mellea.stdlib.requirements import req, simple_validate


@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic. Do not hulucinate.

    Also do not actually search the web. This is a dummy implementation.
    Just return the give response.
    
    Args:
        query: The search query.
    """
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


@tool
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression.
    
    Args:
        expression: An arithmetic expression.
    """
    allowed = set("0123456789 +-*/(). ")
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression))  # noqa: S307


# Initialize session
m: mellea.MelleaSession = mellea.start_session()

# Add requirements to constrain the output format
response = m.instruct(
    "What is Mellea, and how many characters are in the word 'Mellea'?",
    model_options={ModelOption.TOOLS: [web_search, calculate]},
    requirements=[
        # LLM-as-a-judge validates this subjective requirement
        req("The response must answer both questions."),
        # Deterministic validation for objective word count check
        req(
            "The response must be 50 words or fewer.",
            validation_fn=simple_validate(
                lambda x: (
                    len(x.split()) <= 50,
                    f"Response is {len(x.split())} words; must be ≤50."
                )
            ),
        ),
    ],
)

print(str(response))

# Output will vary — LLM responses depend on model and temperature.

## Step 3: Inspecting failures and handling a retry budget

Use `RejectionSamplingStrategy` with `return_sampling_results=True` to observe what happens when requirements fail.

**Key Concepts:**

- `RejectionSamplingStrategy` controls how many times Mellea will retry if requirements fail
- `loop_budget` sets the maximum number of generation attempts (default is 3)
- `return_sampling_results=True` changes the return type from `ModelOutputThunk` to `SamplingResult`
- `SamplingResult` provides:
  - `.success`: Boolean indicating if requirements were satisfied
  - `.result`: The successful output (if success=True)
  - `.sample_generations`: List of all attempts made, each with validation results
- This gives you programmatic control over fallback behavior when requirements cannot be satisfied
- You can inspect each attempt's validation failures to understand what went wrong
- Useful for debugging or for choosing the best available output when the budget runs out

In [ ]:
import mellea
from mellea.backends import ModelOption, tool
from mellea.stdlib.requirements import req, simple_validate
from mellea.stdlib.sampling import RejectionSamplingStrategy


@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic.
    
    Args:
        query: The search query.
    """
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


@tool(name="calculator")
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression.
    
    Args:
        expression: An arithmetic expression.
    """
    allowed = set("0123456789 +-*/(). ")
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression))  # noqa: S307


# Initialize session
m: mellea.MelleaSession = mellea.start_session()

# Configure rejection sampling with explicit retry budget
result = m.instruct(
    "What is Mellea, and how many characters are in the word 'Mellea'?",
    model_options={ModelOption.TOOLS: [web_search, calculate]},
    requirements=[
        req("The response must answer both questions."),
        req(
            "The response must be 50 words or fewer.",
            validation_fn=simple_validate(
                lambda x: (
                    len(x.split()) <= 50,
                    f"Response is {len(x.split())} words; must be ≤50."
                )
            ),
        ),
    ],
    strategy=RejectionSamplingStrategy(loop_budget=3),
    return_sampling_results=True,
)

# Check if any attempt satisfied all requirements
if result.success:
    print("Passed:", str(result.result))
else:
    # All attempts failed - inspect the attempts for debugging
    print(f"Failed after {len(result.sample_generations)} attempts")
    
    # You could implement fallback logic here:
    # - Use the best attempt based on partial validation
    # - Return an error message
    # - Retry with relaxed requirements
    # For now, just show the first attempt
    print("First attempt:", str(result.sample_generations[0].value))

# The word-count requirement runs deterministically. The "answer both questions" requirement 
# falls back to LLM-as-a-judge. If either fails, Mellea retries with the failure reason 
# embedded in the repair request — useful for debugging or for choosing the best available 
# output when the budget runs out.

## Step 4: Adding Guardian harm detection

`GuardianCheck` wraps a `MelleaSession` call and evaluates the output against a set of `GuardianRisk` categories. Run it after your agent responds to flag outputs before they reach downstream code.

**Key Concepts:**

- `GuardianCheck` is a post-generation safety layer that evaluates LLM outputs
- It checks outputs against predefined `GuardianRisk` categories (HARM, PROFANITY, JAILBREAK, etc.)
- Guardian runs as an independent inference call against your local model
- Each check returns a pass/fail result with a reason if it fails
- `backend_type` specifies which LLM backend to use for Guardian checks (e.g., "ollama")
- `backend_id` identifies the specific model (e.g., "granite3-guardian:2b")
- Guardian checks are independent of your main agent logic - they don't modify the output
- Use `m.validate()` to run multiple Guardian checks and get detailed results
- This pattern lets you detect and handle harmful content before it reaches users

Each `GuardianCheck` runs as an independent inference call against your local model. `m.validate()` evaluates the checks against the most recent session output. Run it immediately after the `instruct()` call before any other session activity modifies the context.

In [ ]:
import mellea
from mellea.backends import ModelOption, tool, model_ids
from mellea.backends.ollama import OllamaModelBackend
from mellea.stdlib.requirements import req, simple_validate
from mellea.stdlib.requirements.safety.guardian import GuardianCheck, GuardianRisk
from mellea.stdlib.sampling import RejectionSamplingStrategy


@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic.
    
    Args:
        query: The search query.
    """
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


@tool(name="calculator")
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression.
    
    Args:
        expression: An arithmetic expression.
    """
    allowed = set("0123456789 +-*/(). ")
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression))  # noqa: S307


# Initialize session
m: mellea.MelleaSession = mellea.start_session()

# Generate response with requirements
response = m.instruct(
    "What is Mellea, and how many characters are in the word 'Mellea'?",
    model_options={ModelOption.TOOLS: [web_search, calculate]},
    requirements=[
        req("The response must answer both questions."),
        req(
            "The response must be 50 words or fewer.",
            validation_fn=simple_validate(
                lambda x: (
                    len(x.split()) <= 50,
                    f"Response is {len(x.split())} words; must be ≤50."
                )
            ),
        ),
    ],
    strategy=RejectionSamplingStrategy(loop_budget=3),
)

output_text: str = str(response)

# Run Guardian checks on the agent output
# GuardianCheck evaluates the output against specific risk categories
harm_check = GuardianCheck(
    GuardianRisk.HARM,  # Check for harmful content
    backend_type="ollama",  # Use Ollama backend for Guardian
    model_version="granite3-guardian:2b",  # Specific Guardian model
)

# Validate returns a list of check results
validation_results = m.validate([harm_check])

# Check if all safety checks passed
safe: bool = all(r._result for r in validation_results)

if safe:
    print("Output passed safety checks:", output_text)
else:
    # Handle unsafe output - log, reject, or modify
    for check_result in validation_results:
        if not check_result._result:
            print(f"Safety check failed — {check_result._reason}")

# Output will vary — LLM responses depend on model and temperature.

## Step 5: Optimizing with a shared Guardian backend

When you run multiple `GuardianCheck` instances, each one loads or contacts the model separately by default. Pass `backend=shared_backend` to reuse a single loaded backend and avoid the overhead of repeated initialisation.

**Key Concepts:**

- By default, each `GuardianCheck` creates its own backend instance
- This means loading the Guardian model multiple times, which is slow and memory-intensive
- `OllamaModelBackend` (or other backends) can be created once and shared
- Pass the shared backend to multiple `GuardianCheck` instances via the `backend` parameter
- This significantly improves performance when running multiple safety checks
- The shared backend pattern applies to any Mellea backend, not just Guardian
- Pull the Guardian model first using: `ollama pull granite3-guardian:2b`
- `model_ids.IBM_GRANITE_GUARDIAN` is a constant for the Guardian model identifier

In [ ]:
import mellea
from mellea.backends import ModelOption, tool, model_ids
from mellea.backends.ollama import OllamaModelBackend
from mellea.stdlib.requirements import req, simple_validate
from mellea.stdlib.requirements.safety.guardian import GuardianCheck, GuardianRisk
from mellea.stdlib.sampling import RejectionSamplingStrategy


@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic.
    
    Args:
        query: The search query.
    """
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


@tool(name="calculator")
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression.
    
    Args:
        expression: An arithmetic expression.
    """
    allowed = set("0123456789 +-*/(). ")
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression))  # noqa: S307


# Initialize session
m: mellea.MelleaSession = mellea.start_session()

response = m.instruct(
    "What is Mellea, and how many characters are in the word 'Mellea'?",
    model_options={ModelOption.TOOLS: [web_search, calculate]},
    requirements=[
        req("The response must answer both questions."),
        req(
            "The response must be 50 words or fewer.",
            validation_fn=simple_validate(
                lambda x: (
                    len(x.split()) <= 50,
                    f"Response is {len(x.split())} words; must be ≤50."
                )
            ),
        ),
    ],
    strategy=RejectionSamplingStrategy(loop_budget=3),
)

output_text: str = str(response)

# Create a single Guardian backend and reuse it across all checks
# Pull the model first: ollama pull granite3-guardian:2b
guardian_backend = OllamaModelBackend(model_ids.IBM_GRANITE_GUARDIAN)

# Create multiple Guardian checks that share the same backend
# This avoids loading the model multiple times
checks = [
    GuardianCheck(GuardianRisk.HARM, backend=guardian_backend),
    GuardianCheck(GuardianRisk.PROFANITY, backend=guardian_backend),
    GuardianCheck(GuardianRisk.ANSWER_RELEVANCE, backend=guardian_backend),
    GuardianCheck(GuardianRisk.JAILBREAK, backend=guardian_backend),
]

# Run all checks efficiently with the shared backend
results = m.validate(checks)

# Iterate through results to check each risk category
for risk, result in zip(checks, results):
    status: str = "PASS" if result._result else "FAIL"
    print(f"{risk._risk.name}: {status}")
    if not result._result:
        print(f"  Reason: {result._reason}")

# Output will vary — LLM responses depend on model and temperature.

## Step 6: Groundedness checks with retrieved context

When your agent retrieves documents before answering, add a `GROUNDEDNESS` check to confirm the response is grounded in what was retrieved rather than hallucinated.

**Key Concepts:**

- Groundedness checks verify that LLM responses are based on provided context, not hallucinated
- `GuardianRisk.GROUNDEDNESS` is a specific risk category for detecting hallucinations
- Pass retrieved context via the `grounding_context` parameter in `instruct()`
- The context should be a dictionary with a key (e.g., "docs") mapping to the retrieved text
- Guardian evaluates whether the response is supported by the provided context
- This is crucial for RAG (Retrieval-Augmented Generation) applications
- Use the same `context_text` for both the agent's generation and Guardian's validation
- The `retrieve_docs` tool would normally call a vector store or search index
- In production, replace the mock context with actual retrieved documents

**Tip:** Pass the same text you supplied as `grounding_context` to `context_text` in `GuardianCheck`. This ensures the groundedness model evaluates the response against exactly what the agent was given.

In [ ]:
import mellea
from mellea.backends import ModelOption, tool, model_ids
from mellea.backends.ollama import OllamaModelBackend
from mellea.stdlib.requirements.safety.guardian import GuardianCheck, GuardianRisk


@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic.
    
    Args:
        query: The search query.
    """
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


@tool(name="calculator")
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression.
    
    Args:
        expression: An arithmetic expression.
    """
    allowed = set("0123456789 +-*/(). ")
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression))  # noqa: S307


# Mock retrieved context - in production, this would come from a vector store
RETRIEVED_CONTEXT = (
    "Mellea is an open-source Python framework for building generative AI applications. "
    "It provides instruct(), @generative, and @mify as its core primitives. "
    "Mellea is backend-agnostic and supports Ollama, OpenAI, and custom backends."
)

# Initialize session
m: mellea.MelleaSession = mellea.start_session()

# Generate response with grounding context
# The LLM receives the context and should base its answer on it
response = m.instruct(
    "Using the retrieved documentation, describe what Mellea is.",
    model_options={ModelOption.TOOLS: [web_search, calculate]},
    grounding_context={"docs": RETRIEVED_CONTEXT},
)

output_text: str = str(response)

# Check the response is grounded in the retrieved context
# Pass the same context to Guardian for validation
groundedness_check = GuardianCheck(
    GuardianRisk.GROUNDEDNESS,
    backend_type="ollama",
    model_version="granite3-guardian:2b",
    context_text=RETRIEVED_CONTEXT,
)

# Validate with the same context used for generation
results = m.validate([groundedness_check])

if results[0]._result:
    print("Grounded response:", output_text)
else:
    print("Response may contain hallucinated content.")
    print("Reason:", results[0]._reason)

# Output will vary — LLM responses depend on model and temperature.

## Step 7: A ReACT agent with Guardian checks

Combine everything into a goal-driven agent that uses the ReACT (Reason + Act) loop with Guardian validation.

**Key Concepts:**

- `react()` implements the Reason + Act loop: the LLM alternates between reasoning and tool invocation
- The agent continues until it determines the goal is satisfied or the step budget runs out
- `ChatContext()` maintains conversational state across reasoning steps
- `backend=m.backend` reuses the main session's backend for the agent
- `tools=[web_search, calculate]` provides the agent with callable tools
- Guardian checks run after the agent completes to validate the final output
- This pattern separates agent logic (ReACT loop) from safety validation (Guardian)
- You can inspect intermediate steps via the second return value (the trace list)
- For fine-grained control over each reasoning step, build a custom loop using `m.instruct()` with `ModelOption.TOOLS`

In [ ]:
import asyncio

import mellea
from mellea.backends import ModelOption, tool, model_ids
from mellea.backends.ollama import OllamaModelBackend
from mellea.stdlib.context import ChatContext
from mellea.stdlib.frameworks.react import react
from mellea.stdlib.requirements.safety.guardian import GuardianCheck, GuardianRisk


@tool
def web_search(query: str) -> str:
    """Search the web for information about a topic.
    
    Args:
        query: The search query.
    """
    return f"Top result for '{query}': Mellea is a Python framework for building generative AI applications."


@tool(name="calculator")
def calculate(expression: str) -> str:
    """Evaluate a safe arithmetic expression.
    
    Args:
        expression: An arithmetic expression.
    """
    allowed = set("0123456789 +-*/(). ")
    if not all(c in allowed for c in expression):
        return "Error: expression contains disallowed characters."
    return str(eval(expression))  # noqa: S307


# Define the agent function using ReACT pattern
async def run_agent(goal: str) -> str:
    """Run a ReACT agent with Guardian validation.
    
    Args:
        goal: The goal for the agent to achieve.
        
    Returns:
        The agent's final response after validation.
    """
    # ReACT implements the Reason + Act loop
    # The LLM alternates between reasoning ("Thought") and invoking tools ("Action")
    # until it determines the goal is satisfied or the step budget runs out
    result, _ = await react(
        goal=goal,
        context=ChatContext(),  # Maintains conversation state
        backend=m.backend,  # Reuse the session's backend
        tools=[web_search, calculate],  # Available tools
    )
    return str(result)


# Initialize session
m: mellea.MelleaSession = mellea.start_session()

# Run the agent
output = asyncio.run(run_agent(
    "Find out what Mellea is, then calculate how many characters are in the word 'Mellea'."
))

# Validate the agent's final output with Guardian
harm_check = GuardianCheck(
    GuardianRisk.HARM,
    backend_type="ollama",
    model_version="granite3-guardian:2b",
)

results = m.validate([harm_check])

if results[0]._result:
    print("Agent output:", output)
else:
    print("Agent output flagged:", results[0]._reason)

# Output will vary — LLM responses depend on model and temperature.

## Advanced

`react()` implements the Reason + Act loop: the LLM alternates between producing a reasoning step ("Thought") and invoking a tool ("Action") until it determines the goal is satisfied or the step budget runs out. You can inspect the intermediate steps via the second return value (the trace list). For fine-grained control over each reasoning step, build a custom loop using `m.instruct()` with `ModelOption.TOOLS` directly.

## What you built

| Pattern | What it gives you |
|---------|-------------------|
| `@tool` + `ModelOption.TOOLS` | LLM can invoke Python functions as tools |
| `req()` + `simple_validate()` | Enforce output format and content constraints |
| `RejectionSamplingStrategy` | Explicit retry budget |
| `return_sampling_results=True` | Inspect every attempt for debugging |
| `GuardianCheck` | Post-generation safety risk detection |
| Shared `backend` | Amortise model loading across multiple checks |
| `GuardianRisk.GROUNDEDNESS` + `context_text` | Detect hallucination relative to retrieved context |
| `react()` | Goal-driven multi-step agentic loop |